<a href="https://colab.research.google.com/github/inoue0426/llm-tuning-playground/blob/main/notebooks/03_pharma_dpo_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 - DPO on a real pharmaceutical preference dataset

This notebook uses the public `ThakrePranjal/pharma-preference-dataset` and applies LoRA + DPO to Qwen2.5-0.5B-Instruct.

The goal is educational: move from toy preference pairs in Notebook 02 to a small real-domain preference dataset. This is not clinical validation.


## Pipeline

`public pharma preference data -> conversational format -> Qwen + LoRA -> DPO -> held-out evaluation`

The dataset is small, so the training run is intentionally lightweight.


In [ ]:
!pip -q install -U \
    "transformers>=4.55,<5" \
    "datasets>=3.6,<5" \
    "peft>=0.17,<1" \
    "trl>=0.21,<1" \
    "accelerate>=1.10,<2" \
    "bitsandbytes>=0.46,<1" \
    "torchao>=0.16,<1"


In [ ]:
import torch
import transformers, datasets, peft, trl

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before running this notebook.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


## 1. Load the pharmaceutical preference dataset

Each example is expected to contain `prompt`, `chosen`, and `rejected`.


In [ ]:
from datasets import load_dataset

RAW_DATASET = "ThakrePranjal/pharma-preference-dataset"
raw = load_dataset(RAW_DATASET, split="train")
print(raw)
print(raw.column_names)
print(raw[0])


## 2. Convert to TRL conversational preference format

Qwen is an instruction/chat model. We therefore represent the prompt as a user message and the two answers as assistant messages. TRL can then apply Qwen's chat template consistently during DPO preprocessing.


In [ ]:
from datasets import Dataset

def normalize_prompt(text):
    text = str(text)
    if "### Instruction:" in text:
        text = text.split("### Instruction:", 1)[1]
    if "### Response:" in text:
        text = text.split("### Response:", 1)[0]
    return text.strip()

rows = []
for row in raw:
    rows.append({
        "prompt": [{"role": "user", "content": normalize_prompt(row["prompt"])}],
        "chosen": [{"role": "assistant", "content": str(row["chosen"]).strip()}],
        "rejected": [{"role": "assistant", "content": str(row["rejected"]).strip()}],
    })

dataset = Dataset.from_list(rows)
split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print("Total:", len(dataset))
print("Train:", len(train_dataset))
print("Eval:", len(eval_dataset))
print("Example:", train_dataset[0])


## 3. Load Qwen2.5-0.5B-Instruct

We keep the same base model as Notebooks 01 and 02 so the preference dataset is the main change.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype)
model = model.cuda()

print("Has chat template:", tokenizer.chat_template is not None)


## 4. Baseline generation on a held-out prompt


In [ ]:
def generate(model, question, max_new_tokens=120):
    messages = [{"role": "user", "content": question}]
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    encoded = {k: v.to(model.device) for k, v in encoded.items()}
    with torch.no_grad():
        output = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

TEST_INDEX = 0
test_prompt = eval_dataset[TEST_INDEX]["prompt"][0]["content"]
print("QUESTION:", test_prompt)
print()
print("BEFORE DPO:")
print(generate(model, test_prompt))


## 5. Run DPO with LoRA

The base model is frozen and only a low-rank adapter is trained. `max_prompt_length` is intentionally omitted because the installed TRL 0.29.x API does not accept it in `DPOConfig`.


In [ ]:
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

dpo_args = DPOConfig(
    output_dir="./outputs/pharma-dpo",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=1e-5,
    beta=0.1,
    max_length=512,
    logging_steps=2,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
)

trainer = DPOTrainer(
    model=model,
    args=dpo_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.train()


## 6. Evaluate preference behavior

DPO evaluation metrics such as chosen/rejected rewards and preference accuracy are more informative than inspecting a single generated answer.


In [ ]:
metrics = trainer.evaluate()
print("Evaluation metrics:")
for key, value in metrics.items():
    if isinstance(value, (int, float)):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")


In [ ]:
model.eval()
print()
print("AFTER DPO:")
print(generate(model, test_prompt))


## 7. Inspect several held-out prompts

Read the outputs critically. Fluent text is not evidence of factual correctness.


In [ ]:
for i in range(min(5, len(eval_dataset))):
    question = eval_dataset[i]["prompt"][0]["content"]
    print("=" * 80)
    print("QUESTION:", question)
    print(generate(model, question, max_new_tokens=100))


## 8. Save the LoRA adapter


In [ ]:
ADAPTER_DIR = "./outputs/pharma-dpo-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved:", ADAPTER_DIR)


## Interpretation

This experiment demonstrates how the same DPO machinery behaves with a real pharmaceutical preference dataset. It does **not** establish biomedical factual accuracy. For a more scientifically meaningful experiment, use a licensed biomedical knowledge source to construct evidence-grounded preference pairs and evaluate factual correctness separately.
